# Limiar × Aprendizado de Máquina na detecção de quedas: quanto cada um perde ao trocar de base?

**Projeto Mova (UNAMA).** Estudo 100% offline, só com bases públicas. Não tem app nem coleta com voluntários.

### Pergunta de pesquisa
> Quando um detector de quedas é ajustado ou treinado numa base de dados e testado em outra (*cross-dataset*), os **algoritmos de limiar** perdem desempenho **tanto quanto** um **modelo leve de aprendizado de máquina** (TinyML)?

### Hipóteses
- **H1:** o modelo de ML tem desempenho maior *dentro* da base de treino, mas cai **mais** ao mudar de base. Os limiares começam piores e são mais estáveis.
- **H0:** a queda de desempenho (Δ) é igual nos dois tipos de método.
- **H1-alternativa:** o ML continua melhor e cai menos. É o que Duan et al. (2026) afirmam: queda de cerca de 1% da KFall para a SisFall.

Qualquer um dos três resultados responde a pergunta e serve para o artigo.

### Métodos comparados
| Família | Método | Referência | Fonte da implementação |
|---|---|---|---|
| Limiar | Kangas, variante 2 | Kangas et al. 2008, *Gait & Posture* 28(2) | Parâmetros do Bagalà 2012 e do porte aberto "Guardian" (ver `docs/PIPELINE_ESTUDO_QUEDAS.md`) |
| Limiar | Bourke3 (velocidade + impacto + postura) | Bourke et al. 2010, *J. Biomech.* 43(15) | Parâmetros do Bagalà et al. 2012, *PLoS ONE* 7(5):e37062 |
| Limiar | PIPTO | Moutsis et al. 2023, *Sensors* 23(18):7951 | **Código original dos autores**, baixado do GitHub na execução |
| ML (TinyML) | CNN-LSTM, 21.434 parâmetros | Duan et al. 2026, *Int. J. Technology* 17(3):919-935 | Reimplementada a partir da Tabela 1 e da Seção 4 do artigo; não há código público |

### Desenho experimental
1. **Dentro da base (KFall):**
   - limiares com os parâmetros publicados;
   - limiares reajustados na KFall, com validação cruzada por sujeito;
   - CNN-LSTM com LOSO (deixa um sujeito de fora por vez), como no artigo.
2. **Entre bases (KFall → SisFall):** nada é reajustado nem retreinado; o mesmo detector é aplicado na SisFall.
3. **Métrica principal:** **Δ = desempenho na KFall − desempenho na SisFall**, com intervalo de confiança por *bootstrap* de sujeitos.
4. **Avaliação por tentativa (arquivo)**, igual para todos. A SisFall só rotula o arquivo inteiro como queda ou ADL, sem o instante da queda.

### Como usar
- **Teste rápido:** rode com `MODO_DEMO = True`. O notebook gera dados **sintéticos** no formato das duas bases e executa tudo de ponta a ponta em poucos minutos. **Os números do modo demo não significam nada**; servem só para ver se o código roda.
- **Execução real:**
  1. Baixe as bases (Seção 2).
  2. Coloque `MODO_DEMO = False`.
  3. Rode no Google Colab com GPU. O LOSO completo da CNN-LSTM (32 folds × 25 épocas) leva tempo; para testes, use `LOSO_SUJEITOS = 4`.

In [ ]:
# Dependências (no Colab só falta o openpyxl; localmente, descomente a linha abaixo)
# !pip install numpy scipy pandas matplotlib openpyxl tensorflow

## 1. Configuração
Todos os parâmetros do experimento ficam nesta célula. As escolhas que **não estão nos artigos** aparecem marcadas com `SUPOSIÇÃO`. Elas precisam ser declaradas na seção de Métodos do artigo.

In [ ]:
from pathlib import Path
import os, re, sys, math, json, subprocess, warnings
from dataclasses import dataclass, field
from fractions import Fraction

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample_poly, butter, lfilter, lfilter_zi

warnings.filterwarnings('ignore', category=FutureWarning)

MODO_DEMO = True            # True = dados sintéticos só para testar o código

BASE_DIR   = Path('.').resolve()
DATA_DIR   = BASE_DIR / 'data'
SISFALL_DIR = DATA_DIR / 'SisFall'   # pasta com SA01..SA23, SE01..SE15
KFALL_DIR   = DATA_DIR / 'KFall'     # pasta com sensor_data/ e label_data/
VENDOR_DIR  = BASE_DIR / 'vendor'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

FS      = 50       # Hz: taxa comum a todos os métodos (a CNN-LSTM de Duan usa 50 Hz)
CLIP_G  = 8.0      # SUPOSIÇÃO: satura em ±8 g para simular o acelerômetro de um celular (None = desliga)
G0      = 9.80665
SEED    = 42

# Limiares
PAD_FIM_S = 3.0    # SUPOSIÇÃO: repete a última amostra por 3 s, para a checagem de postura não ser cortada no fim do arquivo
N_FOLDS_AJUSTE = 5 # validação cruzada por sujeito para os limiares reajustados na KFall

# CNN-LSTM (Duan et al. 2026, Seções 4.2 e 4.3)
JANELA_S, PASSO_S = 4.0, 2.0
EPOCAS     = 25
BATCH      = 64                        # SUPOSIÇÃO: o artigo não informa o tamanho do lote
PESO_CLASSE = {0: 0.8, 1: 0.2}         # do artigo: 0,8 para ADL e 0,2 para queda (escolha intencional, ver p. 924-925)
LOSO_SUJEITOS = None                   # None = todos os 32 sujeitos da KFall; um número (ex.: 4) = só os primeiros N
N_SEMENTES_CRUZADO = 3                 # treinos completos na KFall com sementes diferentes, para o teste na SisFall
K_JANELAS = (2, 1)                     # regra por tentativa: queda se houver >= k janelas positivas seguidas (o artigo usa k=2)

if MODO_DEMO:
    DATA_DIR = BASE_DIR / 'data_demo'
    SISFALL_DIR, KFALL_DIR = DATA_DIR / 'SisFall', DATA_DIR / 'KFall'
    EPOCAS, LOSO_SUJEITOS, N_SEMENTES_CRUZADO = 20, 3, 1

rng_global = np.random.default_rng(SEED)
print('MODO_DEMO =', MODO_DEMO, '| dados em', DATA_DIR)

## 2. Bases de dados

### SisFall (teste entre bases)
- **Referência:** Sucerquia, López & Vargas-Bonilla, *Sensors* 2017, 17(1):198.
- **Sensores e taxa:** dispositivo na cintura, 200 Hz. O notebook usa o acelerômetro ADXL345 (±16 g, 13 bits); a conversão é `g = bruto × 32/8192`.
- **Participantes:** 23 jovens (SA01-SA23) e 15 idosos (SE01-SE15). Os idosos fizeram só ADLs, com exceção do SE06.
- **Atividades:** 15 tipos de queda (F01-F15) e 19 de ADL (D01-D19).
- **Onde baixar:**
  - O site oficial (`sistemic.udea.edu.co`) costuma estar fora do ar.
  - Alternativa 1: material suplementar do artigo no MDPI.
  - Alternativa 2: espelho em CSV no GitHub, sem licença declarada:
    ```bash
    git clone https://github.com/krishna-karthikk/SisFall-Dataset data/SisFall_mirror
    ```
    Depois aponte `SISFALL_DIR` para `data/SisFall_mirror/Fall Detection csv`. O espelho tem **4.398 arquivos**, contra 4.505 tentativas do original (o SA07, por exemplo, está incompleto). Declare isso no artigo.
- O carregador aceita os dois formatos: o `.txt` original e o `.csv` do espelho, com cabeçalho `acc1_x,...`.

### KFall (treino e "dentro da base")
- **Referência:** Yu, Jang & Xiong, *Frontiers in Aging Neuroscience* 2021, 13:692865.
- **Sensores e taxa:** IMU na lombar, 100 Hz, aceleração já em **g**.
- **Participantes:** 32 jovens (SA06-SA38).
- **Tarefas:** T01-T19 e T35-T36 são ADL; **T20-T34 são quedas** (F01-F15).
- **Arquivos:**
  - sinais: `sensor_data/SAxx/SxxTyyRzz.csv`, colunas `TimeStamp(s), FrameCounter, AccX..`;
  - rótulos: `label_data/SAxx_label.xlsx`, com `Fall_onset_frame` (início) e `Fall_impact_frame` (impacto).
- **Acesso:** sob **pedido** em https://sites.google.com/view/kfalldataset. Peçam hoje, porque a liberação pode demorar.

> ⚠️ **Conferir na base real:** se os números `Fall_onset_frame` e `Fall_impact_frame` usam a escala da coluna `FrameCounter` (padrão do carregador) ou o número da linha. A Seção 4 imprime um teste que ajuda a decidir.

In [ ]:
# ------------------------- Gerador de dados sintéticos (só MODO_DEMO) -------------------------
def _rot_to(u, v, s):
    '''interpola linearmente entre os vetores u e v (s de 0 a 1) e normaliza'''
    w = (1 - s)[:, None] * u + s[:, None] * v
    return w / np.linalg.norm(w, axis=1, keepdims=True)

def sintetico(tipo, fs, dur, rng, up, lado):
    n = int(dur * fs); t = np.arange(n) / fs
    grav = np.tile(up, (n, 1)).astype(float)
    amp = np.ones(n)
    t0 = rng.uniform(0.35, 0.55) * dur
    i0 = int(t0 * fs)
    if tipo == 'andar':
        amp += 0.25 * np.sin(2 * np.pi * rng.uniform(1.6, 2.0) * t)
    elif tipo in ('sentar_forte', 'pular'):
        d = int((0.25 if tipo == 'sentar_forte' else 0.3) * fs)
        amp[i0:i0 + d] = 0.65 if tipo == 'sentar_forte' else 0.3
        pk = rng.uniform(1.8, 2.6) if tipo == 'sentar_forte' else rng.uniform(2.8, 3.8)
        amp[i0 + d:i0 + d + max(2, int(0.06 * fs))] = pk
    elif tipo in ('queda', 'queda_suave'):
        d = int(rng.uniform(0.3, 0.45) * fs)
        amp[i0:i0 + d] = 0.2 if tipo == 'queda' else 0.55
        pk = rng.uniform(3.0, 5.5) if tipo == 'queda' else rng.uniform(1.6, 2.2)
        k = i0 + d
        amp[k:k + max(2, int(0.06 * fs))] = pk
        r = max(2, int(0.5 * fs))
        s = np.clip((np.arange(n) - (i0 - int(0.2 * fs))) / r, 0, 1)
        grav = _rot_to(np.tile(up, (n, 1)), np.tile(lado, (n, 1)), s)
    acc = grav * amp[:, None] + rng.normal(0, 0.03, (n, 3))
    return acc, (i0 if tipo.startswith('queda') else None), ((i0 + int(0.35 * fs)) if tipo.startswith('queda') else None)

def gerar_demo(root, rng):
    if (root / 'OK').exists():
        return
    adl_tipos = ['andar', 'andar', 'sentar_forte', 'pular']
    # SisFall: 200 Hz, contagens brutas do ADXL345, formato do espelho em CSV
    sis = root / 'SisFall'
    for subj in ['SA01', 'SA02', 'SA03', 'SA04', 'SE01', 'SE06']:
        (sis / subj).mkdir(parents=True, exist_ok=True)
        up, lado = np.array([0, -1.0, 0]), np.array([0, 0, 1.0])
        ativ = [(f'D{d:02d}', adl_tipos[d % 4]) for d in range(1, 9)]
        if subj.startswith('SA') or subj == 'SE06':
            ativ += [(f'F{f:02d}', 'queda' if f % 4 else 'queda_suave') for f in range(1, 7)]
        for cod, tipo in ativ:
            for rep in (1, 2):
                acc, _, _ = sintetico(tipo, 200, 15, rng, up, lado)
                raw = np.round(acc / (32 / 8192))
                df = pd.DataFrame(np.hstack([raw, np.zeros((len(raw), 6))]),
                                  columns=['acc1_x', 'acc1_y', 'acc1_z', 'gyro_x', 'gyro_y', 'gyro_z', 'acc2_x', 'acc2_y', 'acc2_z'])
                df.to_csv(sis / subj / f'{cod}_{subj}_R{rep:02d}.csv', index=False)
    # KFall: 100 Hz, g, rótulos em xlsx
    kf = root / 'KFall'
    for sn in range(6, 12):
        subj = f'SA{sn:02d}'
        (kf / 'sensor_data' / subj).mkdir(parents=True, exist_ok=True)
        (kf / 'label_data').mkdir(parents=True, exist_ok=True)
        up, lado = np.array([1.0, 0, 0]), np.array([0, 0, -1.0])
        rows = []
        tarefas = [(t, adl_tipos[t % 4]) for t in range(1, 9)] + [(t, 'queda' if t % 4 else 'queda_suave') for t in range(20, 26)]
        for task, tipo in tarefas:
            for rep in (1, 2):
                acc, on, im = sintetico(tipo, 100, 12, rng, up, lado)
                n = len(acc)
                df = pd.DataFrame({'TimeStamp(s)': np.arange(n) / 100, 'FrameCounter': np.arange(n) + 1,
                                   'AccX': acc[:, 0], 'AccY': acc[:, 1], 'AccZ': acc[:, 2],
                                   'GyrX': 0.0, 'GyrY': 0.0, 'GyrZ': 0.0, 'EulerX': 0.0, 'EulerY': 0.0, 'EulerZ': 0.0})
                df.to_csv(kf / 'sensor_data' / subj / f'S{sn:02d}T{task:02d}R{rep:02d}.csv', index=False)
                if task >= 20:
                    rows.append({'Task Code (Task ID)': f'F{task-19:02d} ({task})' if rep == 1 else None,
                                 'Description': tipo if rep == 1 else None, 'Trial ID': rep,
                                 'Fall_onset_frame': on + 1, 'Fall_impact_frame': im + 1})
        pd.DataFrame(rows).to_excel(kf / 'label_data' / f'{subj}_label.xlsx', index=False)
    (root / 'OK').write_text('ok')

if MODO_DEMO:
    gerar_demo(DATA_DIR, np.random.default_rng(SEED))
    print('Dados sintéticos prontos em', DATA_DIR)

## 3. Carregamento
Cada tentativa (arquivo) vira um objeto `Trial`:
- aceleração em **g, com a gravidade incluída**, nos 3 eixos;
- taxa de amostragem original;
- rótulo (queda ou ADL), sujeito, grupo (jovem/idoso) e código da atividade;
- na KFall, também o instante do início e do impacto da queda.

In [ ]:
@dataclass
class Trial:
    dataset: str; subject: str; group: str; activity: str; rep: int
    is_fall: bool; fs: float; acc: np.ndarray
    onset: int | None = None; impact: int | None = None   # índices de amostra na taxa fs

SISFALL_ADXL = 32 / 8192   # contagens -> g (ADXL345, ±16 g, 13 bits)
RE_SIS = re.compile(r'^([DF]\d{2})_(S[AE]\d{2})_R(\d{2})$')

def _ler_sisfall(path):
    with open(path) as f:
        primeira = f.readline()
    header = 0 if re.match(r'^\s*[A-Za-z]', primeira) else None
    df = pd.read_csv(path, header=header, skipinitialspace=True, usecols=[0, 1, 2])
    return df.to_numpy(dtype=float) * SISFALL_ADXL

def carregar_sisfall(root):
    trials = []
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() not in ('.txt', '.csv'):
            continue
        m = RE_SIS.match(p.stem)
        if not m:
            continue
        act, subj, rep = m.group(1), m.group(2), int(m.group(3))
        try:
            acc = _ler_sisfall(p)
        except Exception as e:
            print('pulando', p.name, e); continue
        trials.append(Trial('SisFall', subj, 'idoso' if subj.startswith('SE') else 'jovem',
                            act, rep, act.startswith('F'), 200.0, acc))
    return trials

RE_KF = re.compile(r'^S(\d+)T(\d+)R(\d+)$')
KFALL_QUEDAS = set(range(20, 35))   # T20-T34 = F01-F15

def _rotulos_kfall(root):
    lab = {}
    for x in Path(root).rglob('*_label.xlsx'):
        subj = x.name.split('_')[0]
        df = pd.read_excel(x)
        df['Task Code (Task ID)'] = df['Task Code (Task ID)'].ffill()
        for _, r in df.iterrows():
            try:
                task = int(str(r['Task Code (Task ID)']).split('(')[1].split(')')[0])
                trial = int(r['Trial ID'])
            except (IndexError, ValueError):
                continue
            on, im = r.get('Fall_onset_frame'), r.get('Fall_impact_frame')
            if pd.notna(on) and pd.notna(im):
                lab[(subj, task, trial)] = (int(on), int(im))
    return lab

def carregar_kfall(root, frame_escala='FrameCounter'):
    lab = _rotulos_kfall(root)
    trials, sem_rotulo = [], 0
    for p in sorted(Path(root).rglob('S*T*R*.csv')):
        m = RE_KF.match(p.stem)
        if not m:
            continue
        sn, task, rep = (int(g) for g in m.groups())
        subj = f'SA{sn:02d}'
        df = pd.read_csv(p)
        acc = df[['AccX', 'AccY', 'AccZ']].to_numpy(dtype=float)
        is_fall = task in KFALL_QUEDAS
        on = im = None
        if is_fall:
            if (subj, task, rep) not in lab:
                sem_rotulo += 1; continue      # queda sem rótulo de início/impacto: fica de fora
            fon, fim = lab[(subj, task, rep)]
            if frame_escala == 'FrameCounter' and 'FrameCounter' in df:
                fc = df['FrameCounter'].to_numpy()
                on, im = int(np.searchsorted(fc, fon)), int(np.searchsorted(fc, fim))
            else:
                on, im = fon, fim
            on, im = min(on, len(acc) - 1), min(im, len(acc) - 1)
        act = f'F{task-19:02d}' if is_fall else f'T{task:02d}'
        trials.append(Trial('KFall', subj, 'jovem', act, rep, is_fall, 100.0, acc, on, im))
    if sem_rotulo:
        print(f'[KFall] {sem_rotulo} tentativas de queda sem rótulo foram deixadas de fora')
    return trials

sisfall = carregar_sisfall(SISFALL_DIR)
kfall   = carregar_kfall(KFALL_DIR)

def resumo(trials):
    df = pd.DataFrame([dict(base=t.dataset, sujeito=t.subject, grupo=t.group, queda=t.is_fall) for t in trials])
    return df.groupby(['base', 'grupo', 'queda']).agg(tentativas=('sujeito', 'size'), sujeitos=('sujeito', 'nunique'))

display(pd.concat([resumo(kfall), resumo(sisfall)]))

### Checagem de unidades
Os algoritmos de limiar usam valores em **g com a gravidade incluída**. Com a pessoa parada, a magnitude tem de dar ≈ 1 g. Se der ≈ 9,8, os dados estão em m/s² e a conversão está errada.

In [ ]:
def checar_unidades(trials, nome):
    med = np.median([np.median(np.linalg.norm(t.acc[: int(t.fs)], axis=1)) for t in trials])
    print(f'{nome}: mediana de |a| no 1º segundo = {med:.3f} g')
    assert 0.6 < med < 1.4, f'{nome}: unidade suspeita (esperado ≈ 1 g)'

checar_unidades(kfall, 'KFall'); checar_unidades(sisfall, 'SisFall')

# Teste do rótulo da KFall: o pico de |a| deve ficar perto do 'impacto' rotulado
dif = [(np.argmax(np.linalg.norm(t.acc, axis=1)) - t.impact) / t.fs for t in kfall if t.is_fall]
print(f'KFall: distância mediana entre o pico e o impacto rotulado = {np.median(dif):+.2f} s  '
      '(perto de 0 indica que a escala dos frames está certa)')

# Orientação dos eixos: em qual eixo fica a gravidade com a pessoa em pé, em cada base?
for nome, tr in (('KFall', kfall), ('SisFall', sisfall)):
    v = np.median([t.acc[: int(t.fs)].mean(0) for t in tr], axis=0)
    print(f'{nome}: vetor gravidade mediano no 1º segundo (x, y, z) = {np.round(v, 2)}')

> **Por que a orientação dos eixos importa:**
> - Os **limiares** usam a magnitude |a|, que não depende de como o sensor está virado. A postura usa uma referência medida por sujeito.
> - A **CNN-LSTM** recebe ax, ay e az **crus**. Se o eixo vertical for diferente entre KFall e SisFall, o modelo vê na SisFall sinais "girados" que nunca viu no treino.
> - Duan et al. não dizem se alinharam os eixos. Registrem o que a célula acima mostrar na base real.
> - **No modo demo, as duas bases sintéticas têm o eixo vertical diferente de propósito**, para mostrar esse efeito.

## 4. Pré-processamento comum
O mesmo pré-processamento vale para todos os métodos, para a comparação ser justa:
1. **Reamostrar para 50 Hz** com `resample_poly`, que já inclui o filtro anti-*aliasing*. Fica 200→50 Hz na SisFall e 100→50 Hz na KFall.
2. **Saturar em ±8 g**, para simular o acelerômetro de um celular. A SisFall original vai até ±16 g.
3. Converter os índices de início e impacto da KFall para a nova taxa.

In [ ]:
def para_fs(acc, fs_in, fs_out=FS):
    if fs_in == fs_out:
        out = acc.copy()
    else:
        fr = Fraction(int(fs_out), int(fs_in)).limit_denominator(1000)
        out = resample_poly(acc, fr.numerator, fr.denominator, axis=0, padtype='line')
    if CLIP_G:
        out = np.clip(out, -CLIP_G, CLIP_G)
    return out

def preprocessar(trials):
    out = []
    for t in trials:
        r = FS / t.fs
        out.append(Trial(t.dataset, t.subject, t.group, t.activity, t.rep, t.is_fall, float(FS),
                         para_fs(t.acc, t.fs),
                         None if t.onset is None else int(round(t.onset * r)),
                         None if t.impact is None else int(round(t.impact * r))))
    return out

kf50, sis50 = preprocessar(kfall), preprocessar(sisfall)

# Referência de "em pé" por sujeito (usada na postura de Kangas e Bourke3).
# SUPOSIÇÃO: mediana da direção da gravidade no 1º segundo de todas as tentativas do sujeito
# (a maioria começa em pé). Os artigos originais usavam um eixo do sensor preso ao corpo.
def ref_em_pe(trials):
    refs = {}
    for s in sorted({t.subject for t in trials}):
        v = np.array([t.acc[: FS].mean(0) for t in trials if t.subject == s])
        v = v / np.linalg.norm(v, axis=1, keepdims=True)
        m = np.median(v, axis=0)
        refs[s] = m / np.linalg.norm(m)
    return refs

UP = {**{('KFall', s): u for s, u in ref_em_pe(kf50).items()},
      **{('SisFall', s): u for s, u in ref_em_pe(sis50).items()}}

i = next(k for k, t in enumerate(sis50) if t.is_fall)
t = sis50[i]; tt = np.arange(len(t.acc)) / FS
plt.figure(figsize=(9, 2.5)); plt.plot(tt, np.linalg.norm(t.acc, axis=1))
plt.title(f'Exemplo: {t.dataset} {t.subject} {t.activity}: magnitude |a| a 50 Hz'); plt.xlabel('s'); plt.ylabel('g'); plt.show()

## 5. Algoritmos de limiar

Os três algoritmos olham o **formato da série temporal**, e não um valor isolado:
1. **Queda livre:** a magnitude da aceleração cai abaixo de cerca de 0,6 g.
2. **Impacto:** em seguida vem um pico acima de 2 a 3 g.
3. **Postura e repouso:** depois, a gravidade aponta para o lado (pessoa deitada) ou o sinal fica parado perto de 1 g.

Cada função recebe a aceleração a 50 Hz e devolve a lista de **alarmes** (índices de amostra).

### 5.1 Kangas et al. 2008, variante 2 (porte "Guardian")
- **Início da queda:** a magnitude total `SV_TOT` cai abaixo de 0,6 g.
- **Impacto:** acontece em até 1 s depois do início. Basta **qualquer um** destes quatro:
  - `SV_TOT ≥ 2,0 g` (magnitude total);
  - `SV_D ≥ 1,7 g` (magnitude do sinal sem gravidade, passa-alta);
  - `SV_MaxMin ≥ 2,0 g` (maior variação em 0,1 s);
  - `Z2 ≥ 1,5 g` (estimativa da aceleração vertical).
- **Postura:** 2 s depois do último impacto, a média da componente vertical do sinal passa-baixa em 0,4 s fica ≤ 0,5 g, ou seja, a pessoa está deitada.
- **Filtros:** Butterworth de 2ª ordem com corte em 0,25 Hz, recalculados para 50 Hz.

In [ ]:
def _iir(x, b, a):
    zi = lfilter_zi(b, a)
    return np.column_stack([lfilter(b, a, x[:, k], zi=zi * x[0, k])[0] for k in range(3)])

def _pad_fim(acc):
    n = int(PAD_FIM_S * FS)
    return np.vstack([acc, np.repeat(acc[-1:], n, 0)]) if n else acc

KANGAS_PUB = dict(inicio=0.6, svtot=2.0, svd=1.7, maxmin=2.0, z2=1.5,
                  jan_impacto=1.0, atraso_postura=2.0, jan_postura=0.4, deitado=0.5, refratario=10.0)

def kangas_features(acc, up):
    acc = _pad_fim(acc)
    bl, al = butter(2, 0.25, btype='low', fs=FS)
    bh, ah = butter(2, 0.25, btype='high', fs=FS)
    L, H = _iir(acc, bl, al), _iir(acc, bh, ah)
    svtot, svd = np.linalg.norm(acc, axis=1), np.linalg.norm(H, axis=1)
    w = max(1, int(round(0.1 * FS)))
    pad = np.vstack([np.repeat(acc[:1], w - 1, 0), acc])
    sw = np.lib.stride_tricks.sliding_window_view(pad, w, axis=0)
    svmm = np.linalg.norm(sw.max(-1) - sw.min(-1), axis=1)
    z2 = (svtot ** 2 - svd ** 2 - 1.0) / 2.0
    return dict(svtot=svtot, svd=svd, svmm=svmm, z2=z2, vert=L @ up)

def kangas_detect(f, p=KANGAS_PUB):
    sv, n = f['svtot'], len(f['svtot'])
    hit = (sv >= p['svtot']) | (f['svd'] >= p['svd']) | (f['svmm'] >= p['maxmin']) | (f['z2'] >= p['z2'])
    W, D, P = int(round(p['jan_impacto'] * FS)), int(round(p['atraso_postura'] * FS)), int(round(p['jan_postura'] * FS))
    ft = it = -1; ult = -10 ** 9; alarmes = []
    for i in range(1, n):
        if ft > -1: ft -= 1
        if it > -1: it -= 1
        if sv[i - 1] >= p['inicio'] > sv[i]:
            ft = W
        if ft > -1 and hit[i]:
            it = D
        if it == 0:
            if f['vert'][max(0, i - P + 1): i + 1].mean() <= p['deitado'] and (i - ult) / FS >= p['refratario']:
                alarmes.append(i); ult = i
    return alarmes

### 5.2 Bourke et al. 2010, "Bourke3" (velocidade + impacto + postura)
Parâmetros tirados do Bagalà 2012:
- **Limiares de magnitude:** LFT (inferior) = 0,65 g e UFT (impacto) = 2,8 g.
- **Bordas de tempo:**
  - descida (última vez abaixo de LFT até passar de UFT) ≤ 600 ms;
  - subida (última vez que passa de LFT até passar de UFT) ≤ 350 ms.
- **Velocidade vertical:** ≤ −0,7 m/s. É a integral de (SV − 1 g) até o impacto.
- **Postura:** entre t+1 s e t+3 s após o impacto, ângulo com a vertical > 60° em mais de 75% do tempo.
- **SUPOSIÇÕES (não estão no Bagalà):**
  - a integração da velocidade começa no último cruzamento de 1 g para baixo, no máximo 1 s antes do impacto;
  - a gravidade é estimada com média móvel de 0,5 s;
  - o artigo original usava uma taxa maior; aqui tudo roda a 50 Hz.

In [ ]:
BOURKE_PUB = dict(lft=0.65, uft=2.8, tfe=0.600, tre=0.350, v_th=-0.7, post0=1.0, post1=3.0,
                  angulo=60.0, frac=0.75, lookback=1.0, jan_grav=0.5, usar_bordas=True)

def bourke_features(acc, up, p=BOURKE_PUB):
    acc = _pad_fim(acc)
    n = len(acc); w = max(1, int(round(p['jan_grav'] * FS)))
    cs = np.vstack([np.zeros((1, 3)), np.cumsum(acc, 0)])
    idx = np.arange(n); lo = np.maximum(0, idx - w + 1)
    g = (cs[idx + 1] - cs[lo]) / (idx + 1 - lo)[:, None]
    cosang = (g @ up) / np.maximum(np.linalg.norm(g, axis=1), 1e-9)
    return dict(sv=np.linalg.norm(acc, axis=1), theta=np.degrees(np.arccos(np.clip(cosang, -1, 1))))

def _ultimo(mask_idx, ate, desde=0):
    c = mask_idx[(mask_idx < ate) & (mask_idx >= desde)]
    return int(c[-1]) if len(c) else None

def bourke_detect(f, p=BOURKE_PUB):
    sv, th, n = f['sv'], f['theta'], len(f['sv'])
    abaixo = sv < p['lft']
    sobe = np.where(abaixo[:-1] & ~abaixo[1:])[0] + 1        # sv[j-1] < LFT <= sv[j]
    desce = np.where(~abaixo[:-1] & abaixo[1:])[0] + 1       # sv[j-1] >= LFT > sv[j]
    b1 = sv < 1.0
    desce1g = np.where(~b1[:-1] & b1[1:])[0] + 1
    P0, P1 = int(p['post0'] * FS), int(p['post1'] * FS)
    alarmes, pendente, bloq = [], None, 0
    for i in range(1, n):
        if pendente is not None and i >= pendente + P1:
            jan = th[pendente + P0: pendente + P1 + 1]
            if len(jan) and (jan > p['angulo']).mean() > p['frac']:
                alarmes.append(pendente)
            pendente = None
        if pendente is None and i > bloq and sv[i - 1] <= p['uft'] < sv[i]:
            ti = i
            tup = _ultimo(sobe, ti)
            tdown = _ultimo(desce, (tup + 1) if tup is not None else ti)
            if p['usar_bordas']:
                ok_borda = (tup is not None and tdown is not None and
                            (ti - tdown) / FS <= p['tfe'] and (ti - tup) / FS <= p['tre'])
            else:
                ok_borda = True
            lim = max(0, ti - int(p['lookback'] * FS))
            ancora = tdown if tdown is not None else ti
            s = _ultimo(desce1g, ancora + 1, desde=lim)
            s = lim if s is None else s
            v = np.cumsum((sv[s: ti + 1] - 1.0) * G0 / FS)
            ok_vel = len(v) > 0 and min(0.0, v.min()) <= p['v_th']
            if ok_borda and ok_vel:
                pendente, bloq = ti, ti + P1
            else:
                bloq = ti + int(0.5 * FS)
    return alarmes

### 5.3 PIPTO (Moutsis et al. 2023): código original dos autores
O notebook **baixa e chama a função original** `fall_detection` do repositório dos autores, sem modificar nada. Assim a replicação fica o mais fiel possível. O código não é copiado para o nosso repositório porque **a licença não está declarada**; citem o artigo.

**Como a função é chamada:**
- **Entrada:** magnitude em **m/s²** e tempo em **segundos**, como nos scripts dos autores.
- **Unidade interna de tempo:** em centésimos de segundo. Então "105" = 1,05 s, "85" = 0,85 s e "50" = 0,5 s.
- **Atenção:** o limiar alto usa a **média da gravação inteira** (`max(média, 20) + 10` m/s²). Isso só funciona com o arquivo completo, depois do fato; no celular, em tempo real, teria de ser uma média móvel.
- **Pontuação dos autores:** ≥ 1 detecção numa tentativa de queda = acerto (VP). Aqui seguimos a mesma regra por tentativa, igual para todos os métodos.

> Os parâmetros do PIPTO foram desenvolvidos usando a KFall e a MMsys. Então o desempenho dele **"dentro da KFall" não é independente**. Isso é uma ameaça à validade; declarem no artigo.

In [ ]:
PIPTO_DIR = VENDOR_DIR / 'pipto'
pipto_fall_detection = None
try:
    if not (PIPTO_DIR / 'python' / 'def_fall.py').exists():
        VENDOR_DIR.mkdir(exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/smoutsis/fall_detection_through_acceleration_data', str(PIPTO_DIR)],
                       check=True, capture_output=True)
    sys.path.insert(0, str(PIPTO_DIR / 'python'))
    from def_fall import fall_detection as pipto_fall_detection
    print('PIPTO carregado de', PIPTO_DIR)
except Exception as e:
    print('⚠️ Não foi possível carregar o PIPTO (sem internet?):', e)

PIPTO_ERROS = []
def pipto_detect(acc, nome=''):
    v = np.linalg.norm(acc, axis=1) * 9.807
    df = pd.DataFrame({'time': np.arange(len(v)) / FS, 'v': v})
    try:
        res = pipto_fall_detection(df, len(df), FS)
    except Exception as e:          # ex.: UnboundLocalError em casos de borda do código original
        PIPTO_ERROS.append((nome, repr(e)))
        return []
    return [int(round(r[1] * FS / 100)) for r in res]   # 'entry' (centésimos de s) -> índice da amostra

### 5.4 Rodar os limiares com os parâmetros publicados

In [ ]:
def rodar_limiares(trials, kparams=KANGAS_PUB, bparams=BOURKE_PUB, cache=None):
    linhas = []
    for t in trials:
        up = UP[(t.dataset, t.subject)]
        if cache is not None and id(t) in cache:
            fk, fb = cache[id(t)]
        else:
            fk, fb = kangas_features(t.acc, up), bourke_features(t.acc, up)
            if cache is not None: cache[id(t)] = (fk, fb)
        base = dict(dataset=t.dataset, subject=t.subject, group=t.group, activity=t.activity,
                    rep=t.rep, is_fall=t.is_fall)
        linhas.append({**base, 'metodo': 'Kangas', 'n_alarmes': len(kangas_detect(fk, kparams))})
        linhas.append({**base, 'metodo': 'Bourke3', 'n_alarmes': len(bourke_detect(fb, bparams))})
    return pd.DataFrame(linhas)

def rodar_pipto(trials):
    if pipto_fall_detection is None:
        return pd.DataFrame()
    return pd.DataFrame([dict(dataset=t.dataset, subject=t.subject, group=t.group, activity=t.activity,
                              rep=t.rep, is_fall=t.is_fall, metodo='PIPTO',
                              n_alarmes=len(pipto_detect(t.acc, f'{t.subject}_{t.activity}_{t.rep}')))
                         for t in trials])

CACHE = {}
pred_pub = pd.concat([rodar_limiares(kf50, cache=CACHE), rodar_limiares(sis50, cache=CACHE),
                      rodar_pipto(kf50), rodar_pipto(sis50)], ignore_index=True)
pred_pub['metodo'] = pred_pub['metodo'] + ' (publicado)'
print('erros internos do PIPTO:', len(PIPTO_ERROS))
pred_pub.groupby(['metodo', 'dataset']).n_alarmes.describe()[['count', 'mean', 'max']]

## 6. Métricas (iguais para todos os métodos)
- **Por tentativa:**
  - tentativa de **queda** = acerto (VP) se houver ≥ 1 alarme;
  - tentativa de **ADL** = falso positivo (FP) se houver ≥ 1 alarme.
- **Medidas:** sensibilidade (SE), especificidade (SP), precisão, F1 e acurácia, com **IC de 95% de Wilson** para SE e SP.
- **Δ (queda de desempenho)** = métrica na KFall − métrica na SisFall. O IC vem de **bootstrap de sujeitos**: os sujeitos são sorteados com reposição, separadamente em cada base, 1.000 vezes.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def metricas(df):
    alarme = df.n_alarmes > 0
    tp = int((df.is_fall & alarme).sum()); fn = int((df.is_fall & ~alarme).sum())
    fp = int((~df.is_fall & alarme).sum()); tn = int((~df.is_fall & ~alarme).sum())
    se = tp / (tp + fn) if tp + fn else np.nan
    sp = tn / (tn + fp) if tn + fp else np.nan
    pr = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * pr * se / (pr + se) if pr and se and not np.isnan(pr + se) else 0.0
    return dict(SE=se, SP=sp, Prec=pr, F1=f1, Acc=(tp + tn) / max(1, len(df)),
                SE_IC=wilson(tp, tp + fn), SP_IC=wilson(tn, tn + fp), TP=tp, FN=fn, FP=fp, TN=tn)

def _contagens_por_sujeito(df):
    a = df.n_alarmes > 0
    c = pd.DataFrame({'tp': df.is_fall & a, 'fn': df.is_fall & ~a, 'fp': ~df.is_fall & a, 'tn': ~df.is_fall & ~a})
    return c.groupby(df.subject.values).sum().to_numpy(dtype=float)   # (sujeitos, 4)

def _metrica_de_contagens(c, metrica):
    tp, fn, fp, tn = c
    se = tp / (tp + fn) if tp + fn else np.nan
    sp = tn / (tn + fp) if tn + fp else np.nan
    pr = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * pr * se / (pr + se) if (pr + se) > 0 else 0.0
    return {'SE': se, 'SP': sp, 'Prec': pr, 'F1': f1}[metrica]

def delta_bootstrap(df_in, df_out, metrica='F1', B=1000, seed=SEED):
    '''IC 95% de (métrica na base 1 - métrica na base 2), sorteando sujeitos com reposição em cada base'''
    rng = np.random.default_rng(seed)
    ca, cb = _contagens_por_sujeito(df_in), _contagens_por_sujeito(df_out)
    ds = [_metrica_de_contagens(ca[rng.integers(0, len(ca), len(ca))].sum(0), metrica) -
          _metrica_de_contagens(cb[rng.integers(0, len(cb), len(cb))].sum(0), metrica) for _ in range(B)]
    return np.nanpercentile(ds, [2.5, 97.5])

def tabela(pred, B=1000):
    linhas = []
    for m, d in pred.groupby('metodo', sort=False):
        a, b = d[d.dataset == 'KFall'], d[d.dataset == 'SisFall']
        if a.empty or b.empty: continue
        ma, mb = metricas(a), metricas(b)
        lo, hi = delta_bootstrap(a, b, 'F1', B)
        linhas.append({'Método': m,
                       'KFall SE': ma['SE'], 'KFall SP': ma['SP'], 'KFall F1': ma['F1'],
                       'SisFall SE': mb['SE'], 'SisFall SP': mb['SP'], 'SisFall F1': mb['F1'],
                       'ΔF1': ma['F1'] - mb['F1'], 'ΔF1 IC95%': f'[{lo:+.3f}, {hi:+.3f}]',
                       'ΔSE': ma['SE'] - mb['SE'], 'ΔSP': ma['SP'] - mb['SP']})
    return pd.DataFrame(linhas)

B_BOOT = 1000
tab_pub = tabela(pred_pub, B_BOOT)
tab_pub.round(3)

## 7. Limiares reajustados na KFall (comparação justa com o ML)
O ML é **treinado** na KFall. Para a comparação ser justa, os limiares também são **reajustados** na KFall:
1. **Grade de valores:**
   - Kangas: fator de escala aplicado aos 4 limiares de impacto × limiar de início da queda;
   - Bourke3: UFT × LFT.
2. **Escolha:** o par com maior F1 na KFall.
3. **Desempenho "dentro da KFall":** validação cruzada por sujeito em 5 partes. Cada sujeito é avaliado com limiares ajustados **sem ele**.
4. **Teste na SisFall:** com os limiares ajustados em **toda** a KFall.

O PIPTO entra só com os parâmetros publicados, porque ajustá-lo exigiria editar o código dos autores.

In [ ]:
GRADE_K = [dict(KANGAS_PUB, inicio=i, svtot=2.0 * s, svd=1.7 * s, maxmin=2.0 * s, z2=1.5 * s)
           for s in (0.6, 0.8, 1.0, 1.2, 1.4, 1.7, 2.0) for i in (0.4, 0.5, 0.6, 0.7, 0.8)]
GRADE_B = [dict(BOURKE_PUB, uft=u, lft=l) for u in (1.8, 2.2, 2.6, 2.8, 3.2, 3.6, 4.0) for l in (0.5, 0.65, 0.8)]

def _feats(t):
    if id(t) not in CACHE:
        up = UP[(t.dataset, t.subject)]
        CACHE[id(t)] = (kangas_features(t.acc, up), bourke_features(t.acc, up))
    return CACHE[id(t)]

def _pred_alg(trials, alg, p):
    rows = []
    for t in trials:
        fk, fb = _feats(t)
        n = len(kangas_detect(fk, p)) if alg == 'Kangas' else len(bourke_detect(fb, p))
        rows.append(dict(dataset=t.dataset, subject=t.subject, group=t.group, activity=t.activity,
                         rep=t.rep, is_fall=t.is_fall, n_alarmes=n))
    return pd.DataFrame(rows)

def ajustar(trials, alg, grade):
    melhor, mf1 = None, -1
    for p in grade:
        f1 = metricas(_pred_alg(trials, alg, p))['F1']
        if f1 > mf1: melhor, mf1 = p, f1
    return melhor, mf1

sujeitos_kf = sorted({t.subject for t in kf50})
folds = np.array_split(np.random.default_rng(SEED).permutation(sujeitos_kf), min(N_FOLDS_AJUSTE, len(sujeitos_kf)))
pred_aj, PARAMS_AJ = [], {}
for alg, grade in (('Kangas', GRADE_K), ('Bourke3', GRADE_B)):
    # dentro da KFall: validação cruzada por sujeito
    for teste in folds:
        tr = [t for t in kf50 if t.subject not in set(teste)]
        te = [t for t in kf50 if t.subject in set(teste)]
        p, _ = ajustar(tr, alg, grade)
        pred_aj.append(_pred_alg(te, alg, p).assign(metodo=f'{alg} (ajustado KFall)'))
    # entre bases: ajusta em toda a KFall e aplica na SisFall
    p, f1 = ajustar(kf50, alg, grade)
    PARAMS_AJ[alg] = p
    pred_aj.append(_pred_alg(sis50, alg, p).assign(metodo=f'{alg} (ajustado KFall)'))
    print(f'{alg}: parâmetros escolhidos em toda a KFall (F1={f1:.3f}):',
          {k: round(v, 3) for k, v in p.items() if k in ('inicio', 'svtot', 'svd', 'maxmin', 'z2', 'uft', 'lft')})
pred_aj = pd.concat(pred_aj, ignore_index=True)

## 8. CNN-LSTM de Duan et al. (2026): reimplementação

### Arquitetura (Tabela 1 do artigo)
| Camada | Saída | Parâmetros |
|---|---|---|
| Entrada | (200, 4) = 4 s × [ax, ay, az, RMS] | 0 |
| Conv2D por canal (×4), 16 filtros | (24, 1, 16) | 544 × 4 |
| Concatenar | (24, 4, 16) | 0 |
| Conv2D, 32 filtros, núcleo (4, 3) | (21, 2, 32) | 6.176 |
| LSTM 32 → LSTM 16 | (42, 32) → (16) | 8.320 + 3.136 |
| Densa 64 → 8 → 2 (softmax) | | 1.088 + 520 + 18 |
| **Total** | | **21.434** |

### Inconsistência na Tabela 1 (declarar no artigo)
A Tabela 1 diz núcleo **(8, 1)** na primeira convolução. Mas com núcleo (8, 1) não dá para chegar ao mesmo tempo à saída **(24, 1, 16)** e a **544 parâmetros**. O núcleo **(33, 1) com passo (7, 1)** reproduz **exatamente** as duas coisas, além do total de 21.434. Usamos essa versão.

### Suposições (não estão no artigo)
- ativação ReLU nas convoluções e camadas densas;
- tamanho do lote = 64.

### Treino (Seções 4.2 e 4.3)
- janelas de 4 s com passo de 2 s, a 50 Hz, normalizadas com z-score;
- na KFall, uma janela é "queda" se cobre ≥ 50% do intervalo [início, impacto + 0,5 s];
- otimizador Adam com taxa 0,001, 25 épocas, entropia cruzada com pesos de classe {ADL: 0,8; queda: 0,2}.

### Avaliação
- **Por janela:** como no artigo, para conferir se a replicação chega perto dos 99,3% de acurácia na KFall.
- **Por tentativa:** queda se houver ≥ k janelas positivas seguidas (k = 2 como no artigo, e k = 1). Esta é a métrica que entra na comparação com os limiares.
- **Normalização z-score de duas formas:**
  - `por_base`: estatísticas de cada base separadamente, como fez o artigo. A SisFall é normalizada com as próprias estatísticas, uma adaptação leve à base de teste que um celular não teria.
  - `treino`: estatísticas só da KFall de treino. É a forma justa.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

JAN, PASSO = int(JANELA_S * FS), int(PASSO_S * FS)

def construir_cnn_lstm():
    inp = layers.Input((JAN, 4))
    x4 = layers.Reshape((JAN, 4, 1))(inp)
    ramos = [layers.Conv2D(16, (33, 1), strides=(7, 1), activation='relu')(
                 layers.Cropping2D(((0, 0), (c, 3 - c)))(x4)) for c in range(4)]
    x = layers.Concatenate(axis=2)(ramos)
    x = layers.Conv2D(32, (4, 3), activation='relu')(x)
    x = layers.Reshape((-1, 32))(x)
    x = layers.LSTM(32, return_sequences=True)(x)
    x = layers.LSTM(16)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(8, activation='relu')(x)
    out = layers.Dense(2, activation='softmax')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    return m

_m = construir_cnn_lstm()
assert _m.count_params() == 21434, _m.count_params()
print('Parâmetros:', _m.count_params(), '(o artigo informa 21.434) ✓')

In [ ]:
def janelas(t):
    x = np.hstack([t.acc, np.linalg.norm(t.acc, axis=1, keepdims=True)])
    if len(x) < JAN:
        x = np.vstack([x, np.repeat(x[-1:], JAN - len(x), 0)])
    ini = np.arange(0, len(x) - JAN + 1, PASSO)
    X = np.stack([x[s:s + JAN] for s in ini]).astype(np.float32)
    y = np.zeros(len(ini), dtype=np.int64)
    if t.is_fall and t.onset is not None:
        a, b = t.onset, t.impact + int(0.5 * FS)
        dur = max(1, b - a)
        sobre = np.clip(np.minimum(ini + JAN, b) - np.maximum(ini, a), 0, None)
        y = (sobre >= 0.5 * dur).astype(np.int64)
    return X, y

def montar(trials):
    Xs, ys, tid = [], [], []
    for k, t in enumerate(trials):
        X, y = janelas(t); Xs.append(X); ys.append(y); tid.append(np.full(len(y), k))
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(tid)

def zstats(X):
    return X.reshape(-1, 4).mean(0), X.reshape(-1, 4).std(0) + 1e-8

def regra_tentativa(prob, k):
    pos = prob >= 0.5; run = best = 0
    for v in pos:
        run = run + 1 if v else 0; best = max(best, run)
    return best >= k

def avaliar_tentativas(model, trials, X, tid, mu, sd, metodo_base):
    prob = model.predict((X - mu) / sd, verbose=0, batch_size=1024)[:, 1]
    rows = []
    for k_trial, t in enumerate(trials):
        pt = prob[tid == k_trial]
        for k in K_JANELAS:
            rows.append(dict(dataset=t.dataset, subject=t.subject, group=t.group, activity=t.activity,
                             rep=t.rep, is_fall=t.is_fall, metodo=f'{metodo_base} k={k}',
                             n_alarmes=int(regra_tentativa(pt, k))))
    return rows, prob

def treinar(X, y, seed):
    tf.keras.utils.set_random_seed(seed)
    m = construir_cnn_lstm()
    m.fit(X, tf.keras.utils.to_categorical(y, 2), epochs=EPOCAS, batch_size=BATCH,
          class_weight=PESO_CLASSE, verbose=0)
    return m

Xk, yk, tk = montar(kf50)
Xs, ys, ts = montar(sis50)
print('KFall: janelas', Xk.shape, '| janelas de queda:', int(yk.sum()))
print('SisFall: janelas', Xs.shape)

### 8.1 LOSO na KFall ("dentro da base")
Cada sujeito é testado por um modelo treinado **sem ele**. O z-score usa só os sujeitos de treino. Este é o passo mais demorado.

In [ ]:
sub_tid = np.array([kf50[i].subject for i in tk])
sujeitos_loso = sujeitos_kf if LOSO_SUJEITOS is None else sujeitos_kf[:LOSO_SUJEITOS]
loso_rows, loso_janela = [], []
for i_s, s in enumerate(sujeitos_loso):
    tr, te = sub_tid != s, sub_tid == s
    mu, sd = zstats(Xk[tr])
    m = treinar((Xk[tr] - mu) / sd, yk[tr], SEED + i_s)
    te_trials = [t for t in kf50 if t.subject == s]
    idx_map = {id(t): j for j, t in enumerate(te_trials)}
    tid_local = np.array([idx_map[id(kf50[i])] for i in tk[te]])
    rows, prob = avaliar_tentativas(m, te_trials, Xk[te], tid_local, mu, sd, 'CNN-LSTM')
    loso_rows += rows
    pred_w = prob >= 0.5; yt = yk[te]
    loso_janela.append(dict(sujeito=s, acc=(pred_w == yt).mean(),
                            rec=(pred_w & (yt == 1)).sum() / max(1, (yt == 1).sum()),
                            prec=(pred_w & (yt == 1)).sum() / max(1, pred_w.sum())))
    print(f'[{i_s+1}/{len(sujeitos_loso)}] {s}: acurácia por janela = {loso_janela[-1]["acc"]:.3f}')
loso_janela = pd.DataFrame(loso_janela)
print('\nMédia por sujeito, por janela (o artigo informa acurácia de 99,3%):')
print(loso_janela[['acc', 'rec', 'prec']].mean().round(4))

### 8.2 KFall → SisFall (entre bases, sem retreinar)
O modelo é treinado com **toda** a KFall, com `N_SEMENTES_CRUZADO` sementes diferentes, e aplicado na SisFall com as duas formas de z-score.

In [ ]:
mu_k, sd_k = zstats(Xk)
mu_s, sd_s = zstats(Xs)
cruz_rows = []
for sem in range(N_SEMENTES_CRUZADO):
    m = treinar((Xk - mu_k) / sd_k, yk, SEED + 100 + sem)
    for modo, (mu, sd) in (('z treino', (mu_k, sd_k)), ('z por base', (mu_s, sd_s))):
        rows, _ = avaliar_tentativas(m, sis50, Xs, ts, mu, sd, f'CNN-LSTM [{modo}]')
        cruz_rows += [dict(r, semente=sem) for r in rows]
    print('semente', sem, 'ok')
cruz = pd.DataFrame(cruz_rows)

# A parte "dentro da KFall" de cada variante é o mesmo LOSO (z sempre com os sujeitos de treino)
loso = pd.DataFrame(loso_rows)
pred_cnn = []
for modo in ('z treino', 'z por base'):
    for k in K_JANELAS:
        nome = f'CNN-LSTM [{modo}] k={k}'
        pred_cnn.append(loso[loso.metodo == f'CNN-LSTM k={k}'].assign(metodo=nome))
        # com várias sementes: alarme se a maioria das sementes disparou
        c = cruz[cruz.metodo == nome].groupby(['dataset', 'subject', 'group', 'activity', 'rep', 'is_fall', 'metodo'],
                                               as_index=False).n_alarmes.mean()
        c['n_alarmes'] = (c.n_alarmes >= 0.5).astype(int)
        pred_cnn.append(c)
pred_cnn = pd.concat(pred_cnn, ignore_index=True)

# variação entre sementes na SisFall (F1 por semente)
print(cruz.groupby(['metodo', 'semente']).apply(lambda d: metricas(d)['F1']).unstack().round(3))

## 9. Resultado principal: quanto cada método perde de uma base para a outra?

In [ ]:
pred_todos = pd.concat([pred_pub, pred_aj, pred_cnn], ignore_index=True)
if LOSO_SUJEITOS is not None:
    print(f'⚠️ LOSO rodou só com {LOSO_SUJEITOS} sujeitos: a parte "KFall" da CNN usa apenas esses sujeitos.')
TAB = tabela(pred_todos, B_BOOT)
TAB.to_csv(RESULTS_DIR / 'tabela_principal.csv', index=False)
pred_todos.to_csv(RESULTS_DIR / 'predicoes_por_tentativa.csv', index=False)
TAB.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(TAB))
ax[0].bar(x - 0.2, TAB['KFall F1'], 0.4, label='KFall (dentro da base)')
ax[0].bar(x + 0.2, TAB['SisFall F1'], 0.4, label='SisFall (outra base)')
ax[0].set_xticks(x, TAB['Método'], rotation=60, ha='right'); ax[0].set_ylabel('F1'); ax[0].legend(); ax[0].set_ylim(0, 1)
ax[0].set_title('F1 por tentativa')
ic = TAB['ΔF1 IC95%'].str.strip('[]').str.split(',', expand=True).astype(float)
ax[1].errorbar(TAB['ΔF1'], x, xerr=[TAB['ΔF1'] - ic[0], ic[1] - TAB['ΔF1']], fmt='o')
ax[1].axvline(0, color='gray', lw=1); ax[1].set_yticks(x, TAB['Método']); ax[1].invert_yaxis()
ax[1].set_xlabel('ΔF1 = KFall − SisFall (IC 95%, bootstrap de sujeitos)'); ax[1].set_title('Queda de desempenho entre bases')
plt.tight_layout(); plt.savefig(RESULTS_DIR / 'fig_delta_f1.png', dpi=200); plt.show()

## 10. Onde cada método erra na SisFall
- **Falsos positivos por tipo de ADL.** Importam especialmente a D11 (tenta levantar e desaba na cadeira) e a D18 (tropeça andando), que são quase-quedas. Duan et al. relatam erros na D10 e na D17.
- **Especificidade em jovens × idosos.**
- **Quedas perdidas por tipo de queda.** Duan et al. relatam a F11 e a F15.

In [ ]:
sis = pred_todos[pred_todos.dataset == 'SisFall'].copy()
sis['alarme'] = sis.n_alarmes > 0
fp_adl = sis[~sis.is_fall].pivot_table(index='activity', columns='metodo', values='alarme', aggfunc='mean')
fn_queda = 1 - sis[sis.is_fall].pivot_table(index='activity', columns='metodo', values='alarme', aggfunc='mean')
sp_grupo = sis[~sis.is_fall].groupby(['metodo', 'group']).alarme.apply(lambda a: 1 - a.mean()).unstack()

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, d, tit in ((ax[0], fp_adl, 'Taxa de falso positivo por ADL'), (ax[1], fn_queda, 'Taxa de queda perdida por tipo')):
    im = a.imshow(d.values, aspect='auto', cmap='Reds', vmin=0, vmax=1)
    a.set_yticks(range(len(d)), d.index); a.set_xticks(range(d.shape[1]), d.columns, rotation=70, ha='right')
    a.set_title(tit); plt.colorbar(im, ax=a)
plt.tight_layout(); plt.savefig(RESULTS_DIR / 'fig_erros_por_atividade.png', dpi=200); plt.show()

fp_adl.to_csv(RESULTS_DIR / 'fp_por_adl_sisfall.csv'); fn_queda.to_csv(RESULTS_DIR / 'fn_por_queda_sisfall.csv')
print('Especificidade na SisFall por grupo (jovem × idoso):'); sp_grupo.round(3)

## 11. Checklist antes de escrever o artigo
1. **Conferir no PDF original** os parâmetros de Kangas e Bourke3 que vieram de fontes secundárias. A lista está em `docs/PIPELINE_ESTUDO_QUEDAS.md`.
2. **Decidir a escala do rótulo da KFall** (`FrameCounter` ou número da linha), usando a checagem da Seção 3.
3. **Rodar o LOSO completo** (`LOSO_SUJEITOS = None`) e ≥ 3 sementes no teste entre bases.
4. **Comparar a reprodução com o publicado:**
   - CNN-LSTM: acurácia por janela na KFall (≈ 99,3%) e acurácia na SisFall (≈ 98,3%).
   - PIPTO: os números do artigo dele.
5. **Declarar na seção de Métodos todas as `SUPOSIÇÃO`** e a correção do núcleo da 1ª convolução: (33, 1) com passo 7.
6. **Ameaças à validade:**
   - só há quedas simuladas por jovens;
   - as duas bases são de cintura ou lombar, não de celular no bolso;
   - os parâmetros do PIPTO foram desenvolvidos com a KFall;
   - a CNN foi reimplementada sem o código dos autores;
   - o espelho da SisFall está incompleto (4.398 de 4.505 tentativas).
7. **Extensões opcionais:**
   - UMAFall: celular no bolso, para medir a passagem da cintura para o bolso;
   - LTMM: vida real de idosos, para contar alarmes falsos por dia.